# CS224 HW1

CS224 HW1

[DAggar 2011](https://arxiv.org/pdf/1011.0686)


1. Sequential prediction tasks violate iid. Why? Consider the robot scenario to make the problem less general. Effects of noniid include distribution drift.
2. How does DAgger fix the iid problem? Trains on $(s, a^*) \sim d_{\pi}$ instead of $(s, a) \sim d_{\pi^*}$


Iteration 0 or during init:
load expert trajectories from .pkl file




The dataset $D$ consists of N state action (s,a) pairs. In behavior cloning $D_{expert}$ comes from expert demonstrations ${s,a_{expert}}$. For a continuous case we are modeling an action such as a torque angle and velocity.

$D=\{(s_i,a_i)\}_{i=1}^N$

and where $d_E$ is the expert state distribution

$(s_i,a_i)\sim d_E(s)\,\pi_E(a\mid s)$


Objective: max likelihood using NLL of expert actions:

$\theta^{*} =\arg\min_\theta \;\; \mathbb E_{(s,a)\sim D}\big[-\log \pi_\theta(a\mid s)\big]$

The Loss fn:

$L(\theta)=\frac1N\sum_{i=1}^N -\log \pi_\theta(a_i\mid s_i)$

and Gradient


$\nabla_\theta L(\theta)= -\frac1N\sum_{i=1}^N \nabla_\theta \log \pi_\theta(a_i\mid s_i)$


If actions discrete; L is cross entropy

$\pi_\theta(a\mid s)=\text{softmax}(f_\theta(s))_a$


Actions continuous: use gaussian.

$\pi_\theta(a\mid s)=N\!\big(a;\mu_\theta(s),\Sigma\big)$


$L(\theta)\propto \mathbb E\big[\|a-\mu_\theta(s)\|^2\big]$

But this suffers from multimodal effects

$\mathbb E[a\mid s]=\sum_k \alpha_k(s)\mu_k(s)$




In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/'Colab Notebooks'


/content/drive/MyDrive/Colab Notebooks


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [ ]:
!pip -q install "gymnasium[mujoco]" stable-baselines3 imageio[ffmpeg]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 25.7 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy
import torch
import time
startTime = time.time()
env = gym.make("Ant-v4")  # physics on CPU

model = PPO(
    policy=MlpPolicy,
    env=env,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    verbose=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

model.learn(total_timesteps=500_000)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, env_id="Ant-v4", filename="ant.mp4", ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(model, filename="ppo_ant.mp4")
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Streaming output truncated to the last 5000 lines.
|    loss                 | 63.9        |
|    n_updates            | 170         |
|    policy_gradient_loss | -0.0303     |
|    std                  | 0.877       |
|    value_loss           | 137         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 74.8        |
|    ep_rew_mean          | -62.9       |
| time/                   |             |
|    fps                  | 411         |
|    iterations           | 19          |
|    time_elapsed         | 94          |
|    total_timesteps      | 38912       |
| train/                  |             |
|    approx_kl            | 0.010266326 |
|    clip_fraction        | 0.104       |
|    clip_range           | 0.2         |
|    entropy_loss         | -10.3       |
|    explained_variance   | 0.584       |
|    learning_rate        | 0.0003      |
|    loss                

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65537) b'The GLFW library is not initialized'
  warnings.warn(message, GLFWError)


FatalError: gladLoadGL error

In [ ]:
from IPython.display import Video
Video(mp4_path, embed=True)

In [ ]:
!pip install -q stable-baselines3 "gymnasium[mujoco]" mujoco "imageio[ffmpeg]"
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 26.0 MB/s eta 0:00:00


In [ ]:
# use all 8 cores

import os
os.environ["MUJOCO_GL"] = "egl"          # use EGL (headless GPU)
os.environ["PYOPENGL_PLATFORM"] = "egl"
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import time
import gymnasium as gym

startTime = time.time()
venv = make_vec_env("Ant-v4", n_envs=8, vec_env_cls=SubprocVecEnv)

model_8cores = PPO(
    "MlpPolicy",
    venv,
    n_steps=512,        # 512*8 = 4096 per update
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    device="cuda",
)

model_8cores.learn(4096*3)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, env_id="Ant-v4", filename="8_cores_ant.mp4", ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(model_8cores, filename="8_cores_ant.mp4")
print("Saved:", mp4_path)
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 53.3     |
|    ep_rew_mean     | -53.6    |
| time/              |          |
|    fps             | 2937     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 52.9        |
|    ep_rew_mean          | -52.3       |
| time/                   |             |
|    fps                  | 2317        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.008290531 |
|    clip_fraction        | 0.078       |
|    clip_range           | 0.2         |
|    entropy_loss         | -11.3       |
|    explained_variance   | -0.00495    |
|    learnin

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Wrote 8_cores_ant.mp4, return=956.9
Saved: 8_cores_ant.mp4
elapsed_time:0.016097609798113505 hrs


In [ ]:
from IPython.display import Video
Video(mp4_path, embed=True)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip install -q stable-baselines3 "gymnasium[mujoco]" mujoco "imageio[ffmpeg]"
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 25.0 MB/s eta 0:00:00


In [ ]:
# use all 8 cores

import os
os.environ["MUJOCO_GL"] = "egl"          # use EGL (headless GPU)
os.environ["PYOPENGL_PLATFORM"] = "egl"
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import time
import gymnasium as gym

startTime = time.time()
venv = make_vec_env("Ant-v4", n_envs=8, vec_env_cls=SubprocVecEnv)

model_8cores = PPO(
    "MlpPolicy",
    venv,
    n_steps=512,        # 512*8 = 4096 per update
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    device="cuda",
)

model_8cores.learn(4096*100)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, filename, env_id="Ant-v4",  ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(policy=model_8cores, filename="4m_8_cores_ant.mp4")
print("Saved:", mp4_path)
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 51.8     |
|    ep_rew_mean     | -55.8    |
| time/              |          |
|    fps             | 1983     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 4096     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 62.3         |
|    ep_rew_mean          | -66.4        |
| time/                   |              |
|    fps                  | 1679         |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0095142005 |
|    clip_fraction        | 0.0896       |
|    clip_range           | 0.2          |
|    entropy_loss         | -11.3        |
|    explained_variance   | -0.0205      |
|    learning_r

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Wrote 4m_8_cores_ant.mp4, return=67.9
Saved: 4m_8_cores_ant.mp4
elapsed_time:0.064930695494016 hrs


In [ ]:
from IPython.display import Video
Video(mp4_path, embed=True)

In [ ]:
# 1000 iterations 10x longer than above .01 hr this isnt right. off by a 0.
# should be 1h
# A100

import os
os.environ["MUJOCO_GL"] = "egl"          # use EGL (headless GPU)
os.environ["PYOPENGL_PLATFORM"] = "egl"
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import time
import gymnasium as gym

startTime = time.time()
venv = make_vec_env("Ant-v4", n_envs=8, vec_env_cls=SubprocVecEnv)

model_8cores = PPO(
    "MlpPolicy",
    venv,
    n_steps=512,        # 512*8 = 4096 per update
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    device="cuda",
)

model_8cores.learn(4096*1000)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, filename, env_id="Ant-v4",  ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(policy=model_8cores, filename="4m_8_cores_ant.mp4")
print("Saved:", mp4_path)
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")


Streaming output truncated to the last 5000 lines.
|    std                  | 0.0905    |
|    value_loss           | 352       |
---------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 950        |
|    ep_rew_mean          | 3.52e+03   |
| time/                   |            |
|    fps                  | 1923       |
|    iterations           | 774        |
|    time_elapsed         | 1647       |
|    total_timesteps      | 3170304    |
| train/                  |            |
|    approx_kl            | 0.04154269 |
|    clip_fraction        | 0.45       |
|    clip_range           | 0.2        |
|    entropy_loss         | 7.91       |
|    explained_variance   | 0.705      |
|    learning_rate        | 0.0003     |
|    loss                 | 132        |
|    n_updates            | 7730       |
|    policy_gradient_loss | -0.0119    |
|    std                  | 0.0906     |
|    valu

In [ ]:
from IPython.display import Video
Video(mp4_path, embed=True)

In [ ]:
# set ent_coef

#ent_coef = 0.01   # try 0.01 → 0.05



# 1000 iterations 10x longer than above .01 hr this isnt right. off by a 1/2. 30m
# A100

import os
os.environ["MUJOCO_GL"] = "egl"          # use EGL (headless GPU)
os.environ["PYOPENGL_PLATFORM"] = "egl"
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import time
import gymnasium as gym

startTime = time.time()
venv = make_vec_env("Ant-v4", n_envs=8, vec_env_cls=SubprocVecEnv)

model_8cores = PPO(
    "MlpPolicy",
    venv,
    n_steps=512,        # 512*8 = 4096 per update
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    ent_coef = 0.01,
    device="cuda",
)

model_8cores.learn(4096*300)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, filename, env_id="Ant-v4",  ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(policy=model_8cores, filename="4m_8_cores_ant.mp4")
print("Saved:", mp4_path)
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")
#4m_8_cores_ant.mp4
#40k_8_cores_ant.mp4
# 300iterations_ent_coef_01_ant.mp4

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    loss                 | 32.3        |
|    n_updates            | 720         |
|    policy_gradient_loss | -0.0362     |
|    std                  | 0.759       |
|    value_loss           | 80.8        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 56.5        |
|    ep_rew_mean          | -38.8       |
| time/                   |             |
|    fps                  | 1914        |
|    iterations           | 74          |
|    time_elapsed         | 158         |
|    total_timesteps      | 303104      |
| train/                  |             |
|    approx_kl            | 0.012454033 |
|    clip_fraction        | 0.137       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.12       |
|    explained_variance   | 0.92        |
|    learning_rate        | 0.0003      |
|    loss                

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Wrote 4m_8_cores_ant.mp4, return=1010.1
Saved: 4m_8_cores_ant.mp4
elapsed_time:0.19200641499625312 hrs


In [ ]:
from IPython.display import Video
Video(mp4_path, embed=True)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
#how to tune the NN?
#ent_coef=0.01
#log_std_init=-0.7
#learning_rate=2.5e-4

# set ent_coef

#ent_coef = 0.01   # try 0.01 → 0.05



# 1000 iterations 10x longer than above .01 hr this isnt right. off by a 1/2. 30m
# A100

import os
os.environ["MUJOCO_GL"] = "egl"          # use EGL (headless GPU)
os.environ["PYOPENGL_PLATFORM"] = "egl"
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import time
import gymnasium as gym

startTime = time.time()
venv = make_vec_env("Ant-v4", n_envs=8, vec_env_cls=SubprocVecEnv)


policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
    log_std_init=-0.5,
)


model_8cores = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=512,        # 512*8 = 4096 per update
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    ent_coef = 0.01,
    learning_rate=2.5e-4,
    device="cuda",
)

model_8cores.learn(4096*150)

import numpy as np
import imageio.v2 as imageio

def rollout_mp4(policy, filename, env_id="Ant-v4",  ep_len=1000, fps=30):
    env = gym.make(env_id, render_mode="rgb_array")
    obs, info = env.reset()

    frames = []
    total_reward = 0.0

    for _ in range(ep_len):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    env.close()

    with imageio.get_writer(filename, fps=fps, codec="libx264") as w:
        for f in frames:
            w.append_data(np.asarray(f, dtype=np.uint8))

    print(f"Wrote {filename}, return={total_reward:.1f}")
    return filename

mp4_path = rollout_mp4(policy=model_8cores, filename="4m_8_cores_ant.mp4")
print("Saved:", mp4_path)
print(f"elapsed_time:{(time.time() - startTime)/3600} hrs")


Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 94.3     |
|    ep_rew_mean     | -29.1    |
| time/              |          |
|    fps             | 2891     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 132         |
|    ep_rew_mean          | -33.2       |
| time/                   |             |
|    fps                  | 2298        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.014050389 |
|    clip_fraction        | 0.181       |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.3        |
|    explained_variance   | -0.0108     |
|    learning_rate        | 0.

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Wrote 4m_8_cores_ant.mp4, return=788.4
Saved: 4m_8_cores_ant.mp4
elapsed_time:0.1024878223074807 hrs


In [ ]:
#150iterations_ant.mp4
from IPython.display import Video
Video(mp4_path, embed=True)

In [ ]:
# --- MUST be before importing gymnasium/mujoco ---
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

!pip -q install "gymnasium[mujoco]" mujoco stable-baselines3 imageio[ffmpeg]

import time, torch, numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

ENV_ID = "Ant-v5"
LOG_DIR = "./ppo_ant_results"
TOTAL_TIMESTEPS = 1_000*1000
N_ENVS = 8

venv = make_vec_env(ENV_ID, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env = make_vec_env(ENV_ID, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
eval_env.obs_rms = venv.obs_rms
eval_env.training = False
eval_env.norm_reward = False

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"{LOG_DIR}/best_model",
    log_path=f"{LOG_DIR}/eval_logs",
    eval_freq=50_000,
    deterministic=True,
    render=False,
)

policy_kwargs = dict(net_arch=dict(pi=[256,256], vf=[256,256]))

model = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
print("done, hours:", (time.time()-start)/3600)

venv.save(f"{LOG_DIR}/vec_normalize.pkl")

# ---- record MP4 (normalize obs manually using venv stats) ----
mp4_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = mp4_env.reset()
frames = []
ret = 0.0

for _ in range(1000):
    frames.append(mp4_env.render())
    norm_obs = venv.normalize_obs(obs)
    action, _ = model.predict(norm_obs, deterministic=True)
    obs, r, terminated, truncated, info = mp4_env.step(action)
    ret += float(r)
    if terminated or truncated:
        break

mp4_env.close()

with imageio.get_writer("final_ant.mp4", fps=30, codec="libx264") as w:
    for f in frames:
        w.append_data(np.asarray(f, dtype=np.uint8))

print("wrote final_ant.mp4, return:", ret)

Using cuda device
Logging to ./ppo_ant_results/PPO_4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 116      |
|    ep_rew_mean     | -124     |
| time/              |          |
|    fps             | 2544     |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 115         |
|    ep_rew_mean          | -122        |
| time/                   |             |
|    fps                  | 2030        |
|    iterations           | 2           |
|    time_elapsed         | 16          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.015729498 |
|    clip_fraction        | 0.187       |
|    clip_range           | 0.2         |
|    entropy_loss         | -11.3       |
|    explained_vari

In [ ]:
#500kiterations_ant.mp4
from IPython.display import Video
Video("final_ant.mp4", embed=True)

In [ ]:
# --- MUST be before importing gymnasium/mujoco ---
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

!pip -q install "gymnasium[mujoco]" mujoco stable-baselines3 imageio[ffmpeg]

import time, torch, numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

ENV_ID = "Hopper"
LOG_DIR = "./ppo_hopper_results"
TOTAL_TIMESTEPS = 1_000*3000
N_ENVS = 8

venv = make_vec_env(ENV_ID, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env = make_vec_env(ENV_ID, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
eval_env.obs_rms = venv.obs_rms
eval_env.training = False
eval_env.norm_reward = False

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"{LOG_DIR}/best_model",
    log_path=f"{LOG_DIR}/eval_logs",
    eval_freq=50_000,
    deterministic=True,
    render=False,
)

policy_kwargs = dict(net_arch=dict(pi=[256,256], vf=[256,256]))

model = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
print("done, hours:", (time.time()-start)/3600)

venv.save(f"{LOG_DIR}/vec_normalize.pkl")

# ---- record MP4 (normalize obs manually using venv stats) ----
mp4_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = mp4_env.reset()
frames = []
ret = 0.0

for _ in range(1000):
    frames.append(mp4_env.render())
    norm_obs = venv.normalize_obs(obs)
    action, _ = model.predict(norm_obs, deterministic=True)
    obs, r, terminated, truncated, info = mp4_env.step(action)
    ret += float(r)
    if terminated or truncated:
        break

mp4_env.close()

with imageio.get_writer("final_hopper.mp4", fps=30, codec="libx264") as w:
    for f in frames:
        w.append_data(np.asarray(f, dtype=np.uint8))

print("wrote final_hopper.mp4, return:", ret)

Using cuda device
Logging to ./ppo_hopper_results/PPO_4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 18.6     |
|    ep_rew_mean     | 13.2     |
| time/              |          |
|    fps             | 2975     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 29.6        |
|    ep_rew_mean          | 30.8        |
| time/                   |             |
|    fps                  | 2320        |
|    iterations           | 2           |
|    time_elapsed         | 14          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.016173007 |
|    clip_fraction        | 0.223       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.23       |
|    explained_v

In [ ]:
from IPython.display import Video
Video("final_hopper.mp4", embed=True)

In [ ]:
# --- MUST be before importing gymnasium/mujoco ---
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

!pip -q install "gymnasium[mujoco]" mujoco stable-baselines3 imageio[ffmpeg]

import time, torch, numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

ENV_ID = "HalfCheetah-v5"
LOG_DIR = "./ppo_HalfCheetah_results"
TOTAL_TIMESTEPS = 1_000*100
N_ENVS = 8

venv = make_vec_env(ENV_ID, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env = make_vec_env(ENV_ID, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
eval_env.obs_rms = venv.obs_rms
eval_env.training = False
eval_env.norm_reward = False

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"{LOG_DIR}/best_model",
    log_path=f"{LOG_DIR}/eval_logs",
    eval_freq=50_000,
    deterministic=True,
    render=False,
)

policy_kwargs = dict(net_arch=dict(pi=[256,256], vf=[256,256]))

model = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
print("done, hours:", (time.time()-start)/3600)

venv.save(f"{LOG_DIR}/vec_normalize.pkl")

# ---- record MP4 (normalize obs manually using venv stats) ----
mp4_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = mp4_env.reset()
frames = []
ret = 0.0

for _ in range(1000):
    frames.append(mp4_env.render())
    norm_obs = venv.normalize_obs(obs)
    action, _ = model.predict(norm_obs, deterministic=True)
    obs, r, terminated, truncated, info = mp4_env.step(action)
    ret += float(r)
    if terminated or truncated:
        break

mp4_env.close()

with imageio.get_writer("HalfCheetah.mp4", fps=30, codec="libx264") as w:
    for f in frames:
        w.append_data(np.asarray(f, dtype=np.uint8))

print("wrote HalfCheetah-v5.mp4, return:", ret)

Using cuda device
Logging to ./ppo_HalfCheetah_results/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -362     |
| time/              |          |
|    fps             | 3430     |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -352        |
| time/                   |             |
|    fps                  | 2564        |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.008635853 |
|    clip_fraction        | 0.0986      |
|    clip_range           | 0.2         |
|    entropy_loss         | -8.53       |
|    explai

In [ ]:
from IPython.display import Video
Video("HalfCheetah.mp4", embed=True)

In [ ]:
%load_ext tensorboard
import os

# Create the log directory if it doesn't exist
log_dir = "./logs/"
os.makedirs(log_dir, exist_ok=True)
%tensorboard --logdir {log_dir}

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy


policy_kwargs = dict(net_arch=[64, 64])


model = PPO(
    "MlpPolicy",
    "Ant-v4",
    verbose=1,
    policy_kwargs=policy_kwargs,
    tensorboard_log=log_dir
)

# As this runs, scroll back up to the previous cell to see the charts
model.learn(total_timesteps=100000)

In [ ]:
# --- MUST be before importing gymnasium/mujoco ---
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

!pip -q install "gymnasium[mujoco]" mujoco stable-baselines3 imageio[ffmpeg]

import time, torch, numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

ENV_ID = "Walker2d-v5"
LOG_DIR = "./ppo_Walker2d-v5_results"
TOTAL_TIMESTEPS = 1_000*100
N_ENVS = 8

venv = make_vec_env(ENV_ID, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env = make_vec_env(ENV_ID, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
eval_env.obs_rms = venv.obs_rms
eval_env.training = False
eval_env.norm_reward = False

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"{LOG_DIR}/best_model",
    log_path=f"{LOG_DIR}/eval_logs",
    eval_freq=50_000,
    deterministic=True,
    render=False,
)

policy_kwargs = dict(net_arch=dict(pi=[256,256], vf=[256,256]))

model = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
print("done, hours:", (time.time()-start)/3600)

venv.save(f"{LOG_DIR}/vec_normalize.pkl")

# ---- record MP4 (normalize obs manually using venv stats) ----
mp4_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = mp4_env.reset()
frames = []
ret = 0.0

for _ in range(1000):
    frames.append(mp4_env.render())
    norm_obs = venv.normalize_obs(obs)
    action, _ = model.predict(norm_obs, deterministic=True)
    obs, r, terminated, truncated, info = mp4_env.step(action)
    ret += float(r)
    if terminated or truncated:
        break

mp4_env.close()

with imageio.get_writer("Walker2d-v5.mp4", fps=30, codec="libx264") as w:
    for f in frames:
        w.append_data(np.asarray(f, dtype=np.uint8))

print("wrote Walker2d-v5.mp4, return:", ret)

Using cuda device
Logging to ./ppo_Walker2d-v5_results/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 17       |
|    ep_rew_mean     | -1.33    |
| time/              |          |
|    fps             | 3047     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 21.2        |
|    ep_rew_mean          | 1.85        |
| time/                   |             |
|    fps                  | 2344        |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.015700446 |
|    clip_fraction        | 0.217       |
|    clip_range           | 0.2         |
|    entropy_loss         | -8.49       |
|    explai

In [ ]:
from IPython.display import Video
Video("Walker2d-v5.mp4", embed=True)

In [ ]:
# --- MUST be before importing gymnasium/mujoco ---
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

!pip -q install "gymnasium[mujoco]" mujoco stable-baselines3 imageio[ffmpeg]

import time, torch, numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

ENV_ID = "Walker2d-v5"
LOG_DIR = "./ppo_Walker2d-v5_results"
TOTAL_TIMESTEPS = 1_000*100
N_ENVS = 8

venv = make_vec_env(ENV_ID, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env = make_vec_env(ENV_ID, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
eval_env.obs_rms = venv.obs_rms
eval_env.training = False
eval_env.norm_reward = False

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"{LOG_DIR}/best_model",
    log_path=f"{LOG_DIR}/eval_logs",
    eval_freq=50_000,
    deterministic=True,
    render=False,
)

policy_kwargs = dict(net_arch=dict(pi=[256,256], vf=[256,256]))

model = PPO(
    "MlpPolicy",
    venv,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
print("done, hours:", (time.time()-start)/3600)

venv.save(f"{LOG_DIR}/vec_normalize.pkl")

# ---- record MP4 (normalize obs manually using venv stats) ----
mp4_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = mp4_env.reset()
frames = []
ret = 0.0

for _ in range(1000):
    frames.append(mp4_env.render())
    norm_obs = venv.normalize_obs(obs)
    action, _ = model.predict(norm_obs, deterministic=True)
    obs, r, terminated, truncated, info = mp4_env.step(action)
    ret += float(r)
    if terminated or truncated:
        break

mp4_env.close()

with imageio.get_writer("Walker2d-v5.mp4", fps=30, codec="libx264") as w:
    for f in frames:
        w.append_data(np.asarray(f, dtype=np.uint8))

print("wrote Walker2d-v5.mp4, return:", ret)

In [ ]:
#500kiterations_ant.mp4
from IPython.display import Video
Video("Walker2d-v5.mp4", embed=True)